# Part 2, Dog Localization

In [ ]:
import numpy as np
import torch
import cv2
import os
import gc
import xml.etree.ElementTree as ET
from tqdm import tqdm
from torchvision import transforms
from torchvision import models
from PIL import Image

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plot_training_history
import matplotlib.pyplot as plt

from src.config import (
    IMG_SIZE_CNN,
    CLASS_NAMES,
    FIGURES_DIR,
    MODELS_DIR,
    LOCALIZATION_DIR,
    OUTPUTS_DIR,
)

from src.utils import (
    load_labeled_images,
    split_data,
    get_pytorch_dataloaders,
    load_test_images,
    generate_submission_csv,
    build_gpu_augmentation,
)

from src.evaluation import compute_metrics, compute_confusion_matrix
from src.visualization import (
    plot_confusion_matrix,
    plot_training_history,
    plot_sample_predictions,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__)
print(f"Using device: {device}")

### Defining our Grad-CAM Model

In [ ]:
class GradCAM_Locator:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        # Register the hooks to grab data during the forward and backward passes
        self.target_layer.register_forward_hook(self.save_activation)
        self.target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output.detach()

    def save_gradient(self, module, grad_input, grad_ouput):
        self.gradients = grad_ouput[0].detach()
    
    #Heat map and bounding box together
    def get_heatmap_and_bbox(self, image_tensor, original_image_shape, threshold=0.0):
        self.model.eval()
        self.gradients = None
        self.activations = None

        # Forward
        output = self.model(image_tensor)
        dog_class_index = 1
        score = output[:, dog_class_index]

        # Backward
        self.model.zero_grad(set_to_none=True)
        score.backward()

        if self.gradients is None or self.activations is None:
            return None, None

        # Grad-CAM
        pooled_gradients = torch.mean(self.gradients, dim=[0, 2, 3])
        activations = self.activations.clone()
        activations *= pooled_gradients.view(1, -1, 1, 1)

        #Heatmap generation
        heatmap = torch.mean(activations, dim=1).squeeze()
        heatmap = F.relu(heatmap)

        max_val = torch.max(heatmap)
        if max_val > 0:
            heatmap = heatmap / max_val

        heatmap = heatmap.detach().cpu().numpy()

        # Resize to original image size
        heatmap_resized = cv2.resize(heatmap, (original_image_shape[1], original_image_shape[0]))

        # Bounding box from heatmap
        binary_map = (heatmap_resized > threshold).astype(np.uint8)

        ys, xs = np.where(binary_map > 0)
        if len(xs) == 0 or len(ys) == 0:
            return heatmap_resized, None

        x_min, x_max = xs.min(), xs.max()
        y_min, y_max = ys.min(), ys.max()

        bbox = [int(x_min), int(y_min), int(x_max - x_min), int(y_max - y_min)]

        return heatmap_resized, bbox

### Loading our model

In [ ]:
model_path = MODELS_DIR / OPTION_C_CNN
# print(model_path)
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
for param in model.parameters():
    param.requires_grad = True
num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 2),
)
model.load_state_dict(torch.load(model_path, weights_only=True, map_location=device))
model.to(device)
model.eval()

### Loading all Image paths

In [ ]:
all_image_paths = []

for breed_folder in os.listdir(PART2_IMAGES_DIR):
    breed_path = PART2_IMAGES_DIR / breed_folder

    if os.path.isdir(breed_path):
        for img_name in os.listdir(breed_path):
            if img_name.endswith(".jpg"):
                all_image_paths.append(breed_path / img_name)

print(f"Found {len(all_image_paths)} images")

### Analysis helper functions

In [ ]:
def compute_iou(boxA, boxB):
    # determine the box = [x, y, w, h]
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
	
	# compute the area of intersection rectangle
    inter_area = max(0, xB - xA) * max(0, yB - yA)

    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]

	# compute union
    union = areaA + areaB - inter_area

    if union == 0:
        return 0.0
    
	# compute the intersection over union
    return inter_area / union

In [ ]:
def load_gt_box(xml_path):
    tree = ET.parse(xml_path)
    root = tree.getroot()

    boxes = []
    # Extract coordinates of each box
    for bbox in root.findall(".//bndbox"):
        xmin = int(bbox.find("xmin").text)
        ymin = int(bbox.find("ymin").text)
        xmax = int(bbox.find("xmax").text)
        ymax = int(bbox.find("ymax").text)
        boxes.append((xmin, ymin, xmax, ymax))

    if not boxes:
        return None

    # Merge all boxes into one
    xmin = min(b[0] for b in boxes)
    ymin = min(b[1] for b in boxes)
    xmax = max(b[2] for b in boxes)
    ymax = max(b[3] for b in boxes)

    return [xmin, ymin, xmax - xmin, ymax - ymin]

In [ ]:
# Given an image path, return the corresponding annotation path
def get_annotation_path(image_path):
    image_path = Path(image_path)
    #Breed
    breeds = image_path.parent.name
    #Filename without extension
    stem = image_path.stem
    return PART2_ANNOTATIONS_DIR / breeds / stem

In [ ]:
# Attach the hooks to the target conv layer
target_layer = model.layer4[2].conv3
cam_extractor = GradCAM_Locator(model, target_layer)

### Quantitative Evaluation

In [ ]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE_CNN)),
    transforms.ToTensor(),
])

In [ ]:
ious = []

for idx, image_path in enumerate(tqdm(all_image_paths)):
    # Load image
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get corresponding annotation path
    xml_path = get_annotation_path(image_path)

    # Load ground truth box from XML
    gt_box = load_gt_box(xml_path)

    # Generate heatmap and predicted box using Grad-CAM
    with torch.enable_grad():
        _, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # Compute IoU and store
    if pred_box is None:
        ious.append(0.0)
    else:
        iou = compute_iou(pred_box, gt_box)
        ious.append(iou)

    # Free memory
    del img, img_rgb, img_tensor

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print("IoUs collected:", len(ious))

In [ ]:
ious = np.array(ious)

# Compute metrics
mean_iou = np.mean(ious)
acc_poor = np.mean(ious < 0.5) * 100
acc_50 = np.mean((ious >= 0.5) & (ious < 0.7)) * 100
acc_70 = np.mean((ious >= 0.7) & (ious < 0.9)) * 100
acc_90 = np.mean(ious >= 0.9) * 100

# Outputs
print(f"Mean IoU: {mean_iou:.4f}")
print(f"Frequency of Poor IoU: {acc_poor:.2f}%")
print(f"Frequency of Average IoU: {acc_50:.2f}%")
print(f"Frequency of Good IoU: {acc_70:.2f}%")
print(f"Frequency of Excellent IoU: {acc_90:.2f}%")

### Qualitative Evaluation of Best and Worse performing dogs

In [ ]:
worst_k = 50
best_k = 50
worst_cases = []
best_cases = []

for image_path in tqdm(all_image_paths):
    # Load image
    img = cv2.imread(str(image_path))
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get corresponding annotation path
    xml_path = get_annotation_path(image_path)

    # Load ground truth box from XML
    gt_box = load_gt_box(xml_path)

    # Extract predicted bounding box
    _,pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # Compute IoU and store
    if pred_box is None:
        iou = 0.0
        pred_box_to_store = None
    else:
        iou = compute_iou(pred_box, gt_box)
        pred_box_to_store = pred_box.copy()   # important

    # Store case details for worst and best cases
    case = {
        "iou": float(iou),
        "image_path": str(image_path),
        "pred_box": pred_box_to_store,
        "gt_box": gt_box.copy()
    }

    # Update worst and best cases
    worst_cases.append(case)
    worst_cases = sorted(worst_cases, key=lambda x: x["iou"])[:worst_k]

    best_cases.append(case)
    best_cases = sorted(best_cases, key=lambda x: x["iou"], reverse=True)[:best_k]

    del img, img_rgb, img_tensor

### Print 50 worse performing cases

In [ ]:
print("=== Worst 50 cases ===")
breed_count = {}
for i, case in enumerate(worst_cases, 1):
    print(f"{i}. IoU={case['iou']:.4f}")
    print(f"   image: {case['image_path']}")
    breed = str(Path(case['image_path']).parent).split('-')[-1]
    print(f"    breed: {breed}")
    if breed not in breed_count:
        breed_count.update({breed: 1})
    else:
        breed_count.update({breed: breed_count[breed]+1})

# Sort by the value (x[1]) in descending order
sorted_breed_count = dict(sorted(breed_count.items(), key=lambda x: x[1], reverse=True))

print(sorted_breed_count)

### Display Worse Bounding Boxes

In [ ]:
plt.figure(figsize=(128, 128))
print(len(worst_cases))
for i, case in enumerate(worst_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    vis = img.copy()

    # Get boxes
    pred_box = case["pred_box"]
    gt_box = case["gt_box"]

    # Draw predicted
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Draw GT
    gx, gy, gw, gh = gt_box
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("50 Worst IoU Cases (Predicted = Green, GT = Red)", fontsize=16)
plt.tight_layout()
plt.show()

### Display corresponding Heat Maps

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(worst_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get predicted bounding box and heatmap
    heatmap, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # overlay heatmap
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap),cv2.COLORMAP_JET)

    # visible image on Display
    vis = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # predicted (green)
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # GT (red)
    gx, gy, gw, gh = case["gt_box"]
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("Worst 50 Cases with Grad-CAM Heatmaps", fontsize=16)
plt.tight_layout()
plt.show()

### Qualitative Evaluation of Best performing Dogs

In [ ]:
print("=== Best 50 cases ===")
breed_count = {}

for i, case in enumerate(best_cases, 1):
    print(f"{i}. IoU={case['iou']:.4f}")
    print(f"   image: {case['image_path']}")
    breed = str(Path(case['image_path']).parent).split('-')[-1]
    print(f"    breed: {breed}")
    if breed not in breed_count:
        breed_count.update({breed: 1})
    else:
        breed_count.update({breed: breed_count[breed]+1})

# Sort by the value (x[1]) in descending order
sorted_breed_count = dict(sorted(breed_count.items(), key=lambda x: x[1], reverse=True))

print(sorted_breed_count)

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(best_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    vis = img.copy()

    # Get boxes
    pred_box = case["pred_box"]
    gt_box = case["gt_box"]

    #Draw predicted
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # Draw GT
    gx, gy, gw, gh = gt_box
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("50 Best IoU Cases (Predicted = Green, GT = Red)", fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(128, 128))

for i, case in enumerate(best_cases[:50]):
    # Load image
    img = cv2.imread(case["image_path"])
    if img is None:
        continue
    
    #Original shape for resizing heatmap later
    original_shape = img.shape[:2]

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Preprocess and prepare tensor for Grad-CAM
    img_tensor = transform(img_rgb).unsqueeze(0).to(device)
    img_tensor.requires_grad_(True)

    # Get predicted bounding box and heatmap
    heatmap, pred_box = cam_extractor.get_heatmap_and_bbox(img_tensor, original_shape, threshold=0)

    # overlay heatmap
    heatmap_color = cv2.applyColorMap(np.uint8(255 * heatmap),cv2.COLORMAP_JET)

    # visible image on Display
    vis = cv2.addWeighted(img, 0.6, heatmap_color, 0.4, 0)

    # predicted (green)
    if pred_box is not None:
        x, y, w, h = pred_box
        cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 255, 0), 2)

    # GT (red)
    gx, gy, gw, gh = case["gt_box"]
    cv2.rectangle(vis, (gx, gy), (gx + gw, gy + gh), (0, 0, 255), 2)

    # Display
    plt.subplot(10, 5, i + 1)
    plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
    plt.title(f"IoU={case['iou']:.2f}")
    plt.axis('off')

plt.suptitle("Best 10 Cases with Grad-CAM Heatmaps", fontsize=16)
plt.tight_layout()
plt.show()